In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
import shap
from scipy import stats
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.problem import Problem
from pymoo.optimize import minimize
from mpl_toolkits.mplot3d import Axes3D
import os
import warnings
warnings.filterwarnings('ignore')

# Set random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Set font and style
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.style.use('seaborn-v0_8-paper')

# Create visualization folder
os.makedirs('visualizations', exist_ok=True)


class ComprehensiveConcreteAnalysis:
    """Comprehensive manufactured-sand concrete analysis system (with multi-model comparison)"""
    
    def __init__(self, data_path='dataset.xlsx'):
        """Initialize the system"""
        print("=" * 80)
        print("Manufactured Sand Concrete Mix Design Intelligent System")
        print("With Multi-Model Comparison")
        print("=" * 80)
        
        self.df = pd.read_excel(data_path)
        self.feature_columns = ['water_cement_ratio', 'cement_grade', 'sand_ratio',
                               'manufactured_sand_ratio', 'stone_powder_content',
                               'stone_powder_type', 'fly_ash_content', 'slag_content',
                               'fineness_modulus']
        self.target_columns = ['strength_7d', 'strength_28d', 'slump', 'flow']
        
        # Model dictionaries
        self.all_models = {}  # All models
        self.scalers = {}
        self.shap_values = {}
        
        # Model name mapping
        self.model_names = {
            'XGBoost': 'XGBoost',
            'SVR': 'Support Vector Regression',
            'MLP': 'Multi-Layer Perceptron',
            'RF': 'Random Forest',
            'GBDT': 'Gradient Boosting'
        }
        
        # Feature name mapping (English)
        self.feature_names_en = {
            'water_cement_ratio': 'W/C Ratio',
            'cement_grade': 'Cement Grade',
            'sand_ratio': 'Sand Ratio',
            'manufactured_sand_ratio': 'M-sand Ratio',
            'stone_powder_content': 'Stone Powder',
            'stone_powder_type': 'Powder Type',
            'fly_ash_content': 'Fly Ash',
            'slag_content': 'Slag',
            'fineness_modulus': 'Fineness Modulus'
        }
        
        print(f"\nData loaded: {len(self.df)} samples")
        print(f"Grade distribution:\n{self.df['grade'].value_counts()}")
    
    def plot_correlation_heatmap(self):
        """Plot the feature correlation heatmap"""
        print("\n[1/18] Plotting correlation heatmap...")
        
        fig, ax = plt.subplots(figsize=(12, 10))
        
        corr_matrix = self.df[self.feature_columns].corr()
        labels = [self.feature_names_en[col] for col in self.feature_columns]
        
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
        sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
                   center=0, square=True, linewidths=1,
                   cbar_kws={"shrink": 0.8, "label": "Correlation Coefficient"},
                   xticklabels=labels, yticklabels=labels,
                   vmin=-1, vmax=1, mask=mask, ax=ax)
        
        ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=20)
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.savefig('visualizations/feature_correlation_heatmap.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Saved: visualizations/feature_correlation_heatmap.png")
    
    def plot_data_distribution(self):
        """Plot the data distribution charts"""
        print("\n[2/18] Plotting data distribution...")
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle('Data Distribution by Concrete Grade', fontsize=16, fontweight='bold')
        
        grades = ['C30', 'C35', 'C40', 'C45']
        colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']
        
        # Water-cement ratio distribution
        ax = axes[0, 0]
        for grade, color in zip(grades, colors):
            data = self.df[self.df['grade'] == grade]['water_cement_ratio']
            ax.hist(data, bins=20, alpha=0.6, label=grade, color=color, edgecolor='black')
        ax.set_xlabel('Water-Cement Ratio')
        ax.set_ylabel('Frequency')
        ax.set_title('W/C Ratio Distribution')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # 28-day strength distribution
        ax = axes[0, 1]
        for grade, color in zip(grades, colors):
            data = self.df[self.df['grade'] == grade]['strength_28d']
            ax.hist(data, bins=20, alpha=0.6, label=grade, color=color, edgecolor='black')
        ax.set_xlabel('28-day Strength (MPa)')
        ax.set_ylabel('Frequency')
        ax.set_title('Strength Distribution')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # Slump distribution
        ax = axes[1, 0]
        for grade, color in zip(grades, colors):
            data = self.df[self.df['grade'] == grade]['slump']
            ax.hist(data, bins=20, alpha=0.6, label=grade, color=color, edgecolor='black')
        ax.set_xlabel('Slump (mm)')
        ax.set_ylabel('Frequency')
        ax.set_title('Slump Distribution')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # Stone powder content distribution
        ax = axes[1, 1]
        for grade, color in zip(grades, colors):
            data = self.df[self.df['grade'] == grade]['stone_powder_content']
            ax.hist(data, bins=20, alpha=0.6, label=grade, color=color, edgecolor='black')
        ax.set_xlabel('Stone Powder Content')
        ax.set_ylabel('Frequency')
        ax.set_title('Stone Powder Distribution')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('visualizations/data_distribution.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Saved: visualizations/data_distribution.png")
    
    def train_all_models(self):
        """Train all models (XGBoost, SVR, MLP, RF, GBDT)"""
        print("\n[3/18] Training all models (5 algorithms × 4 targets)...")
        
        X = self.df[self.feature_columns]
        y = self.df[self.target_columns]
        
        # Stratified split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=RANDOM_SEED,
            stratify=self.df['grade']
        )
        
        print(f"  Training set: {len(X_train)} samples")
        print(f"  Test set: {len(X_test)} samples")
        
        # Store all results
        all_results = {}
        
        for target in self.target_columns:
            print(f"\n  Training models for {target}...")
            
            # Standardization
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            self.scalers[target] = scaler
            
            target_results = {}
            
            # 1. XGBoost
            print("    - XGBoost")
            if target in ['strength_7d', 'strength_28d']:
                xgb_params = {
                    'n_estimators': 300, 'learning_rate': 0.03,
                    'max_depth': 6, 'min_child_weight': 2,
                    'gamma': 0.1, 'subsample': 0.9,
                    'colsample_bytree': 0.8, 'reg_alpha': 0.01,
                    'reg_lambda': 0.1, 'random_state': RANDOM_SEED
                }
            else:
                xgb_params = {
                    'n_estimators': 250, 'learning_rate': 0.05,
                    'max_depth': 5, 'min_child_weight': 3,
                    'gamma': 0.15, 'subsample': 0.85,
                    'colsample_bytree': 0.75, 'reg_alpha': 0.05,
                    'reg_lambda': 0.15, 'random_state': RANDOM_SEED
                }
            xgb_model = xgb.XGBRegressor(**xgb_params)
            xgb_model.fit(X_train_scaled, y_train[target])
            target_results['XGBoost'] = self._evaluate_model(
                xgb_model, X_train_scaled, X_test_scaled, y_train[target], y_test[target]
            )
            
            # 2. SVR
            print("    - SVR")
            svr_model = SVR(kernel='rbf', C=100, gamma='scale', epsilon=0.1)
            svr_model.fit(X_train_scaled, y_train[target])
            target_results['SVR'] = self._evaluate_model(
                svr_model, X_train_scaled, X_test_scaled, y_train[target], y_test[target]
            )
            
            # 3. MLP
            print("    - MLP")
            mlp_model = MLPRegressor(
                hidden_layer_sizes=(100, 50),
                activation='relu',
                solver='adam',
                alpha=0.001,
                batch_size=32,
                learning_rate='adaptive',
                learning_rate_init=0.001,
                max_iter=500,
                random_state=RANDOM_SEED,
                early_stopping=True
            )
            mlp_model.fit(X_train_scaled, y_train[target])
            target_results['MLP'] = self._evaluate_model(
                mlp_model, X_train_scaled, X_test_scaled, y_train[target], y_test[target]
            )
            
            # 4. Random Forest
            print("    - Random Forest")
            rf_model = RandomForestRegressor(
                n_estimators=300,
                max_depth=15,
                min_samples_split=5,
                min_samples_leaf=2,
                random_state=RANDOM_SEED,
                n_jobs=-1
            )
            rf_model.fit(X_train_scaled, y_train[target])
            target_results['RF'] = self._evaluate_model(
                rf_model, X_train_scaled, X_test_scaled, y_train[target], y_test[target]
            )
            
            # 5. GBDT
            print("    - GBDT")
            gbdt_model = GradientBoostingRegressor(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=5,
                min_samples_split=5,
                min_samples_leaf=2,
                subsample=0.8,
                random_state=RANDOM_SEED
            )
            gbdt_model.fit(X_train_scaled, y_train[target])
            target_results['GBDT'] = self._evaluate_model(
                gbdt_model, X_train_scaled, X_test_scaled, y_train[target], y_test[target]
            )
            
            all_results[target] = target_results
            
            # Save the best model (XGBoost) to the main model dictionary
            if target not in self.all_models:
                self.all_models[target] = {}
            self.all_models[target]['XGBoost'] = xgb_model
            self.all_models[target]['SVR'] = svr_model
            self.all_models[target]['MLP'] = mlp_model
            self.all_models[target]['RF'] = rf_model
            self.all_models[target]['GBDT'] = gbdt_model
        
        self.X_train = X_train_scaled
        self.X_test = X_test_scaled
        self.y_train = y_train
        self.y_test = y_test
        self.all_results = all_results
        
        # Save the best model for later SHAP and other analyses
        self.models = {target: self.all_models[target]['XGBoost'] for target in self.target_columns}
        self.results = {target: all_results[target]['XGBoost'] for target in self.target_columns}
        
        return all_results
    
    def _evaluate_model(self, model, X_train, X_test, y_train, y_test):
        """Evaluate a single model"""
        y_pred_test = model.predict(X_test)
        
        # Cross-validation
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
        
        return {
            'test_r2': r2_score(y_test, y_pred_test),
            'rmse': np.sqrt(mean_squared_error(y_test, y_pred_test)),
            'mae': mean_absolute_error(y_test, y_pred_test),
            'cv_mean': cv_scores.mean(),
            'cv_std': cv_scores.std(),
            'y_pred': y_pred_test
        }
    
    def plot_model_comparison_r2(self):
        """Plot R² comparison for all models"""
        print("\n[4/18] Plotting model comparison (R²)...")
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Model Performance Comparison - R² Score', fontsize=16, fontweight='bold')
        
        model_list = ['XGBoost', 'SVR', 'MLP', 'RF', 'GBDT']
        colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#9b59b6']
        
        for idx, target in enumerate(self.target_columns):
            ax = axes[idx // 2, idx % 2]
            
            r2_scores = [self.all_results[target][model]['test_r2'] for model in model_list]
            
            bars = ax.bar(model_list, r2_scores, color=colors, alpha=0.8,
                         edgecolor='black', linewidth=1.5)
            
            # Add value labels
            for bar, score in zip(bars, r2_scores):
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height,
                       f'{score:.4f}', ha='center', va='bottom',
                       fontweight='bold', fontsize=9)
            
            ax.set_ylabel('R² Score', fontweight='bold')
            ax.set_title(f'{target.replace("_", " ").title()}', fontweight='bold')
            # ax.set_ylim([0.85, 1.0])  # <-- Delete or comment out this line
            ax.grid(True, alpha=0.3, axis='y')
            ax.axhline(y=0.95, color='red', linestyle='--', linewidth=1.5, alpha=0.5)
            
            # Mark the best model
            best_idx = np.argmax(r2_scores)
            ax.text(best_idx, r2_scores[best_idx] + 0.01, '★',
                   ha='center', fontsize=20, color='red')
        
        plt.tight_layout()
        plt.savefig('visualizations/model_comparison_R2_score.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Saved: visualizations/model_comparison_R2_score.png")
    
        
    def plot_model_comparison_errors(self):
        """Plot error comparison for all models"""
        print("\n[5/18] Plotting model comparison (RMSE & MAE)...")
        
        fig, axes = plt.subplots(2, 4, figsize=(18, 10))
        fig.suptitle('Model Performance Comparison - Error Metrics', fontsize=16, fontweight='bold')
        
        model_list = ['XGBoost', 'SVR', 'MLP', 'RF', 'GBDT']
        colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#9b59b6']
        
        for idx, target in enumerate(self.target_columns):
            # RMSE
            ax = axes[0, idx]
            rmse_vals = [self.all_results[target][model]['rmse'] for model in model_list]
            bars = ax.bar(model_list, rmse_vals, color=colors, alpha=0.8,
                         edgecolor='black', linewidth=1.5)
            
            for bar, val in zip(bars, rmse_vals):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                       f'{val:.2f}', ha='center', va='bottom',
                       fontweight='bold', fontsize=8)
            
            ax.set_ylabel('RMSE', fontweight='bold')
            ax.set_title(f'{target.replace("_", " ").title()} - RMSE', fontweight='bold')
            ax.grid(True, alpha=0.3, axis='y')
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right')
            
            # MAE
            ax = axes[1, idx]
            mae_vals = [self.all_results[target][model]['mae'] for model in model_list]
            bars = ax.bar(model_list, mae_vals, color=colors, alpha=0.8,
                         edgecolor='black', linewidth=1.5)
            
            for bar, val in zip(bars, mae_vals):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                       f'{val:.2f}', ha='center', va='bottom',
                       fontweight='bold', fontsize=8)
            
            ax.set_ylabel('MAE', fontweight='bold')
            ax.set_title(f'{target.replace("_", " ").title()} - MAE', fontweight='bold')
            ax.grid(True, alpha=0.3, axis='y')
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=15, ha='right')
        
        plt.tight_layout()
        plt.savefig('visualizations/model_comparison_error_metrics.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Saved: visualizations/model_comparison_error_metrics.png")
    
    def plot_model_ranking(self):
        """Plot the overall model ranking"""
        print("\n[6/18] Plotting model ranking...")
        
        model_list = ['XGBoost', 'SVR', 'MLP', 'RF', 'GBDT']
        
        # Compute the average R² for each model
        avg_r2 = {}
        for model in model_list:
            r2_list = [self.all_results[target][model]['test_r2'] 
                      for target in self.target_columns]
            avg_r2[model] = np.mean(r2_list)
        
        # Sort
        sorted_models = sorted(avg_r2.items(), key=lambda x: x[1], reverse=True)
        models_sorted = [m[0] for m in sorted_models]
        scores_sorted = [m[1] for m in sorted_models]
        
        fig, ax = plt.subplots(figsize=(10, 6))
        
        colors_map = {
            'XGBoost': '#2ecc71',
            'SVR': '#3498db',
            'MLP': '#e74c3c',
            'RF': '#f39c12',
            'GBDT': '#9b59b6'
        }
        colors = [colors_map[m] for m in models_sorted]
        
        bars = ax.barh(models_sorted, scores_sorted, color=colors, alpha=0.8,
                      edgecolor='black', linewidth=1.5)
        
        # Add value labels
        for bar, score, rank in zip(bars, scores_sorted, range(1, len(models_sorted)+1)):
            ax.text(score, bar.get_y() + bar.get_height()/2,
                   f' #{rank}  R²={score:.4f}',
                   va='center', fontweight='bold', fontsize=11)
        
        ax.set_xlabel('Average R² Score Across All Targets', fontweight='bold')
        ax.set_title('Overall Model Performance Ranking', fontsize=14, fontweight='bold', pad=15)
        # ax.set_xlim([0.9, 1.0])  # <-- Delete or comment out this line
        ax.grid(True, alpha=0.3, axis='x')
        ax.axvline(x=0.95, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Target (0.95)')
        ax.legend()
        
        plt.tight_layout()
        plt.savefig('visualizations/model_overall_ranking.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Saved: visualizations/model_overall_ranking.png")

    
    def plot_r2_comparison(self):
        """Plot the XGBoost R² comparison (keeps the original functionality)"""
        print("\n[7/18] Plotting XGBoost R² comparison...")
        
        fig, ax = plt.subplots(figsize=(10, 6))
        
        targets = ['strength_7d', 'strength_28d', 'slump', 'flow']
        target_labels = ['7d Strength', '28d Strength', 'Slump', 'Flow']
        r2_scores = [self.results[t]['test_r2'] for t in targets]
        colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']
        
        bars = ax.bar(target_labels, r2_scores, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
        
        for bar, score in zip(bars, r2_scores):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{score:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
        
        ax.set_ylabel('R² Score', fontweight='bold')
        ax.set_title('XGBoost Model Performance - Coefficient of Determination (R²)',
                    fontsize=14, fontweight='bold', pad=15)
        #ax.set_ylim([0.9, 1.0])
        ax.grid(True, alpha=0.3, axis='y')
        ax.axhline(y=0.95, color='red', linestyle='--', linewidth=2, label='Target (R²=0.95)')
        ax.legend()
        
        plt.tight_layout()
        plt.savefig('visualizations/XGBoost_R2_score.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Saved: visualizations/XGBoost_R2_score.png")
    
    def plot_error_metrics(self):
        """Plot XGBoost error metrics"""
        print("\n[8/18] Plotting XGBoost error metrics...")
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle('XGBoost Error Metrics', fontsize=16, fontweight='bold')
        
        targets = ['strength_7d', 'strength_28d', 'slump', 'flow']
        target_labels = ['7d Strength', '28d Strength', 'Slump', 'Flow']
        colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']
        
        # RMSE
        rmse_vals = [self.results[t]['rmse'] for t in targets]
        bars1 = ax1.bar(target_labels, rmse_vals, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
        ax1.set_ylabel('RMSE', fontweight='bold')
        ax1.set_title('Root Mean Square Error', fontweight='bold')
        ax1.grid(True, alpha=0.3, axis='y')
        
        for bar, val in zip(bars1, rmse_vals):
            ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f'{val:.2f}', ha='center', va='bottom', fontweight='bold')
        
        # MAE
        mae_vals = [self.results[t]['mae'] for t in targets]
        bars2 = ax2.bar(target_labels, mae_vals, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
        ax2.set_ylabel('MAE', fontweight='bold')
        ax2.set_title('Mean Absolute Error', fontweight='bold')
        ax2.grid(True, alpha=0.3, axis='y')
        
        for bar, val in zip(bars2, mae_vals):
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f'{val:.2f}', ha='center', va='bottom', fontweight='bold')
        
        plt.tight_layout()
        plt.savefig('visualizations/XGBoost_error_metrics.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Saved: visualizations/XGBoost_error_metrics.png")
    
    def plot_cv_stability(self):
        """Plot cross-validation stability"""
        print("\n[9/18] Plotting cross-validation stability...")
        
        fig, ax = plt.subplots(figsize=(10, 6))
        
        targets = ['strength_7d', 'strength_28d', 'slump', 'flow']
        target_labels = ['7d Strength', '28d Strength', 'Slump', 'Flow']
        cv_means = [self.results[t]['cv_mean'] for t in targets]
        cv_stds = [self.results[t]['cv_std'] for t in targets]
        
        x_pos = np.arange(len(targets))
        ax.errorbar(x_pos, cv_means, yerr=cv_stds, fmt='o-', capsize=8,
                   capthick=3, color='#2c3e50', markersize=10, linewidth=3,
                   markerfacecolor='#3498db', markeredgecolor='black', markeredgewidth=2)
        
        ax.set_xticks(x_pos)
        ax.set_xticklabels(target_labels)
        ax.set_ylabel('CV R² Score', fontweight='bold')
        ax.set_title('XGBoost 5-Fold Cross-Validation Stability',
                    fontsize=14, fontweight='bold', pad=15)
        #ax.set_ylim([0.9, 1.0])
        ax.grid(True, alpha=0.3)
        ax.axhline(y=0.95, color='red', linestyle='--', linewidth=2, alpha=0.5)
        
        for x, mean, std in zip(x_pos, cv_means, cv_stds):
            ax.text(x, mean + std + 0.005, f'{mean:.4f}±{std:.4f}',
                   ha='center', va='bottom', fontsize=9, fontweight='bold')
        
        plt.tight_layout()
        plt.savefig('visualizations/XGBoost_cross_validation_stability.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Saved: visualizations/XGBoost_cross_validation_stability.png")
    
    def plot_prediction_scatter(self):
        """Plot XGBoost prediction scatter plots"""
        print("\n[10/18] Plotting XGBoost prediction scatter plots...")
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 12))
        fig.suptitle('XGBoost: Predicted vs Actual Values', fontsize=16, fontweight='bold')
        
        target_info = [
            ('strength_7d', '7-day Strength (MPa)'),
            ('strength_28d', '28-day Strength (MPa)'),
            ('slump', 'Slump (mm)'),
            ('flow', 'Flow (mm)')
        ]
        
        for idx, (target, label) in enumerate(target_info):
            ax = axes[idx // 2, idx % 2]
            
            y_pred = self.models[target].predict(self.X_test)
            y_actual = self.y_test[target].values
            
            ax.scatter(y_actual, y_pred, alpha=0.6, s=40, edgecolors='black',
                      linewidth=0.5, c='#3498db')
            
            min_val = min(y_actual.min(), y_pred.min())
            max_val = max(y_actual.max(), y_pred.max())
            ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2.5, label='Ideal Line')
            
            r2 = r2_score(y_actual, y_pred)
            rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
            
            ax.set_xlabel(f'Actual {label}', fontweight='bold')
            ax.set_ylabel(f'Predicted {label}', fontweight='bold')
            ax.set_title(f'{label}\nR²={r2:.4f}, RMSE={rmse:.2f}', fontweight='bold')
            ax.legend(loc='upper left')
            ax.grid(True, alpha=0.3)
            
            margin = (max_val - min_val) * 0.1
            ax.fill_between([min_val, max_val],
                           [min_val - margin, max_val - margin],
                           [min_val + margin, max_val + margin],
                           alpha=0.2, color='red', label='±10% Band')
        
        plt.tight_layout()
        plt.savefig('visualizations/XGBoost_prediction_accuracy_scatter.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Saved: visualizations/XGBoost_prediction_accuracy_scatter.png")
    
    def shap_analysis(self):
        """SHAP interpretability analysis"""
        print("\n[11/18] Computing SHAP values...")
        
        for target in ['strength_28d', 'slump']:
            explainer = shap.TreeExplainer(self.models[target])
            shap_values = explainer.shap_values(self.X_test)
            self.shap_values[target] = (shap_values, explainer.expected_value)
        
        print("  ✓ SHAP analysis completed")
    
    def plot_shap_summary(self):
        """Plot SHAP summary plots"""
        print("\n[12/18] Plotting SHAP summary plots...")
        
        for target in ['strength_28d', 'slump']:
            fig, ax = plt.subplots(figsize=(10, 8))
            
            shap_vals, _ = self.shap_values[target]
            feature_names = [self.feature_names_en[col] for col in self.feature_columns]
            
            shap.summary_plot(shap_vals, self.X_test,
                            feature_names=feature_names,
                            show=False, plot_size=None)
            
            title = f'SHAP Summary Plot - {target.replace("_", " ").title()}'
            plt.title(title, fontsize=14, fontweight='bold', pad=15)
            plt.tight_layout()
            
            filename = f'visualizations/SHAP_summary_plot_{target}.png'
            plt.savefig(filename, dpi=300, bbox_inches='tight')
            plt.close()
            print(f"  ✓ Saved: {filename}")
    
    def plot_shap_importance(self):
        """Plot SHAP feature importance"""
        print("\n[13/18] Plotting SHAP feature importance...")
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        fig.suptitle('SHAP Feature Importance', fontsize=16, fontweight='bold')
        
        for idx, target in enumerate(['strength_28d', 'slump']):
            ax = axes[idx]
            
            shap_vals, _ = self.shap_values[target]
            feature_importance = np.abs(shap_vals).mean(0)
            feature_names = [self.feature_names_en[col] for col in self.feature_columns]
            
            sorted_idx = np.argsort(feature_importance)
            pos = np.arange(len(sorted_idx))
            
            ax.barh(pos, feature_importance[sorted_idx], color='#3498db', alpha=0.8,
                   edgecolor='black', linewidth=1)
            ax.set_yticks(pos)
            ax.set_yticklabels([feature_names[i] for i in sorted_idx])
            ax.set_xlabel('Mean |SHAP Value|', fontweight='bold')
            ax.set_title(target.replace('_', ' ').title(), fontweight='bold')
            ax.grid(True, alpha=0.3, axis='x')
            
            for i, v in enumerate(feature_importance[sorted_idx]):
                ax.text(v, i, f' {v:.4f}', va='center', fontsize=9)
        
        plt.tight_layout()
        plt.savefig('visualizations/SHAP_feature_importance.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Saved: visualizations/SHAP_feature_importance.png")


    def plot_shap_dependence(self):
        """Plot SHAP dependence plots - show how individual features affect predictions"""
        print("\\n[Extra] Plotting SHAP dependence plots...")
        
        # Select the four most important features for display
        important_features = ['water_cement_ratio', 'cement_grade', 'stone_powder_content', 'manufactured_sand_ratio']
        
        for target in ['strength_28d', 'slump']:
            fig, axes = plt.subplots(2, 2, figsize=(14, 10))
            fig.suptitle(f'SHAP Dependence Plots - {target.replace("_", " ").title()}', 
                        fontsize=16, fontweight='bold')
            
            shap_vals, _ = self.shap_values[target]
            
            for idx, feature in enumerate(important_features):
                ax = axes[idx // 2, idx % 2]
                feature_idx = self.feature_columns.index(feature)
                
                # Create dependence plot
                shap.dependence_plot(
                    feature_idx, 
                    shap_vals, 
                    self.X_test,
                    feature_names=[self.feature_names_en[col] for col in self.feature_columns],
                    ax=ax,
                    show=False
                )
                
            plt.tight_layout()
            plt.savefig(f'visualizations/SHAP_dependence_plot_{target}.png', dpi=300, bbox_inches='tight')
            plt.close()
            print(f"  ✓ Saved: visualizations/SHAP_dependence_plot_{target}.png")


    def plot_shap_waterfall(self):
        """Plot SHAP waterfall plots - explain individual sample predictions"""
        print("\\n[Extra] Plotting SHAP waterfall plots...")
        
        for target in ['strength_28d', 'slump']:
            shap_vals, base_value = self.shap_values[target]
            predictions = self.models[target].predict(self.X_test)
            
            # Select three representative samples
            indices = [
                np.argmin(predictions),  # lowest prediction
                len(predictions) // 2,   # middle sample
                np.argmax(predictions)    # highest prediction
            ]
            
            for idx, sample_idx in enumerate(indices):
                # Create SHAP explanation object
                shap_explanation = shap.Explanation(
                    values=shap_vals[sample_idx],
                    base_values=base_value,
                    data=self.X_test[sample_idx],
                    feature_names=[self.feature_names_en[col] for col in self.feature_columns]
                )
                
                # Create a separate figure for each sample
                plt.figure(figsize=(10, 6))
                shap.waterfall_plot(shap_explanation, max_display=9, show=False)
                
                pred_val = predictions[sample_idx]
                actual_val = self.y_test[target].iloc[sample_idx]
                
                plt.title(f'{target.replace("_", " ").title()} - Sample {sample_idx}\\n'
                         f'Predicted: {pred_val:.2f}, Actual: {actual_val:.2f}',
                         fontsize=12, fontweight='bold', pad=15)
                
                # Save figure
                sample_type = ['min', 'median', 'max'][idx]
                filename = f'visualizations/SHAP_waterfall_plot_{target}_{sample_type}.png'
                plt.savefig(filename, dpi=300, bbox_inches='tight')
                plt.close()
                print(f"  ✓ Saved: {filename}")

    
    def plot_shap_force(self):
        """Plot SHAP force plots - visualize feature contributions for a batch of samples"""
        print("\\n[Extra] Generating SHAP force plots...")
        
        for target in ['strength_28d', 'slump']:
            shap_vals, base_value = self.shap_values[target]
            
            # Select the first 20 samples for display
            n_samples = 20
            
            # Generate force plot HTML
            force_plot = shap.force_plot(
                base_value,
                shap_vals[:n_samples],
                self.X_test[:n_samples],
                feature_names=[self.feature_names_en[col] for col in self.feature_columns],
                show=False
            )
            
            # Save as an HTML file
            shap.save_html(f'visualizations/SHAP_force_plot_{target}.html', force_plot)
            print(f"  ✓ Saved: visualizations/SHAP_force_plot_{target}.html")


    def plot_shap_bar(self):
        """Plot SHAP bar charts - clearer feature importance display"""
        print("\\n[Extra] Plotting SHAP bar charts...")
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        fig.suptitle('SHAP Global Feature Importance', fontsize=16, fontweight='bold')
        
        for idx, target in enumerate(['strength_28d', 'slump']):
            ax = axes[idx]
            shap_vals, _ = self.shap_values[target]
            
            # Create SHAP bar chart
            shap.summary_plot(
                shap_vals, 
                self.X_test,
                feature_names=[self.feature_names_en[col] for col in self.feature_columns],
                plot_type="bar",
                show=False,
                max_display=9
            )
            
            plt.sca(ax)
            plt.title(target.replace('_', ' ').title(), fontweight='bold')
        
        plt.tight_layout()
        plt.savefig('visualizations/SHAP_bar_plot.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Saved: visualizations/SHAP_bar_plot.png")
        
    def uncertainty_quantification(self):
        """Uncertainty quantification"""
        print("\n[14/18] Computing uncertainty quantification...")
        
        quantile_models = {}
        
        for target in ['strength_28d', 'slump']:
            quantiles = [0.05, 0.50, 0.95]
            quantile_models[target] = {}
            
            for q in quantiles:
                model = GradientBoostingRegressor(
                    loss='quantile', alpha=q, n_estimators=200,
                    max_depth=5, learning_rate=0.05, random_state=RANDOM_SEED
                )
                model.fit(self.X_train, self.y_train[target])
                quantile_models[target][q] = model
        
        self.quantile_models = quantile_models
        print("  ✓ Uncertainty quantification completed")
    
    def plot_uncertainty(self):
        """Plot uncertainty quantification charts"""
        print("\n[15/18] Plotting uncertainty quantification...")
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        fig.suptitle('Prediction Uncertainty Quantification (90% Confidence Interval)',
                    fontsize=16, fontweight='bold')
        
        target_info = [('strength_28d', '28-day Strength (MPa)'),
                      ('slump', 'Slump (mm)')]
        
        for idx, (target, label) in enumerate(target_info):
            ax = axes[idx]
            
            pred_05 = self.quantile_models[target][0.05].predict(self.X_test)
            pred_50 = self.quantile_models[target][0.50].predict(self.X_test)
            pred_95 = self.quantile_models[target][0.95].predict(self.X_test)
            y_actual = self.y_test[target].values
            
            sorted_idx = np.argsort(y_actual)
            x_axis = np.arange(len(sorted_idx))
            
            ax.fill_between(x_axis, pred_05[sorted_idx], pred_95[sorted_idx],
                           alpha=0.3, color='#3498db', label='90% CI')
            ax.plot(x_axis, pred_50[sorted_idx], 'b-', label='Median Prediction',
                   linewidth=2.5)
            ax.scatter(x_axis, y_actual[sorted_idx], s=15, c='red',
                      alpha=0.6, label='Actual Values', edgecolors='black', linewidths=0.5)
            
            ax.set_xlabel('Sample Index (Sorted by Actual Value)', fontweight='bold')
            ax.set_ylabel(label, fontweight='bold')
            ax.set_title(label, fontweight='bold')
            ax.legend(loc='upper left')
            ax.grid(True, alpha=0.3)
            
            coverage = np.mean((y_actual >= pred_05) & (y_actual <= pred_95))
            ax.text(0.98, 0.02, f'Coverage: {coverage:.1%}',
                   transform=ax.transAxes, ha='right', va='bottom',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8),
                   fontsize=10, fontweight='bold')
        
        plt.tight_layout()
        plt.savefig('visualizations/uncertainty_quantification.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Saved: visualizations/uncertainty_quantification.png")
    
    def multi_objective_optimization(self):
        """Multi-objective optimization"""
        print("\n[16/18] Running multi-objective optimization...")
        
        class ConcreteMixProblem(Problem):
            def __init__(self, models, scalers, feature_columns):
                self.models = models
                self.scalers = scalers
                self.feature_columns = feature_columns
                
                xl = np.array([0.35, 42.5, 0.36, 0.40, 0.03, 1, 0.10, 0.10, 2.5])
                xu = np.array([0.46, 52.5, 0.40, 0.80, 0.07, 1, 0.15, 0.20, 3.0])
                
                super().__init__(n_var=9, n_obj=4, n_constr=1, xl=xl, xu=xu)
            
            def _evaluate(self, x, out, *args, **kwargs):
                n_samples = x.shape[0]
                strength_28d = np.zeros(n_samples)
                slump = np.zeros(n_samples)
                
                for i in range(n_samples):
                    sample = x[i:i+1]
                    X_scaled = self.scalers['strength_28d'].transform(sample)
                    strength_28d[i] = self.models['strength_28d'].predict(X_scaled)[0]
                    X_scaled = self.scalers['slump'].transform(sample)
                    slump[i] = self.models['slump'].predict(X_scaled)[0]
                
                binder = 350
                cement = binder * (1 - x[:, 6] - x[:, 7])
                fly_ash = binder * x[:, 6]
                slag = binder * x[:, 7]
                
                cost = (cement * 0.45 + fly_ash * 0.20 + slag * 0.25 +
                       600 * 0.08 + 1100 * 0.07)
                
                carbon = cement * 0.82 + fly_ash * 0.02 + slag * 0.05
                
                f1 = -strength_28d
                f2 = np.abs(slump - 210)
                f3 = cost
                f4 = carbon
                
                out["F"] = np.column_stack([f1, f2, f3, f4])
                out["G"] = 40 - strength_28d
        
        problem = ConcreteMixProblem(self.models, self.scalers, self.feature_columns)
        algorithm = NSGA2(pop_size=100)
        
        res = minimize(problem, algorithm, ('n_gen', 200),
                      seed=RANDOM_SEED, verbose=False)
        
        self.pareto_result = res
        print(f"  ✓ Found {len(res.X)} Pareto optimal solutions")
        
        return res
    
    def plot_pareto_3d(self):
        """Plot Pareto 3D scatter plot"""
        print("\n[17/18] Plotting 3D Pareto front...")
        
        res = self.pareto_result
        strength = -res.F[:, 0]
        slump_error = res.F[:, 1]
        cost = res.F[:, 2]
        carbon = res.F[:, 3]
        
        norm_strength = (strength - strength.min()) / (strength.max() - strength.min())
        norm_slump = 1 - (slump_error - slump_error.min()) / (slump_error.max() - slump_error.min())
        norm_cost = 1 - (cost - cost.min()) / (cost.max() - cost.min())
        norm_carbon = 1 - (carbon - carbon.min()) / (carbon.max() - carbon.min())
        composite_score = norm_strength + norm_slump + norm_cost + norm_carbon
        best_idx = np.argmax(composite_score)
        
        fig = plt.figure(figsize=(12, 9))
        ax = fig.add_subplot(111, projection='3d')
        
        scatter = ax.scatter(strength, cost, carbon, c=slump_error,
                           cmap='viridis', s=50, alpha=0.6, edgecolors='black', linewidth=0.5)
        ax.scatter(strength[best_idx], cost[best_idx], carbon[best_idx],
                  c='red', s=300, marker='*', edgecolors='black', linewidths=2,
                  label='Recommended Solution')
        
        ax.set_xlabel('28-day Strength (MPa)', fontweight='bold', labelpad=10)
        ax.set_ylabel('Cost (CNY/m³)', fontweight='bold', labelpad=10)
        ax.set_zlabel('Carbon Emission (kgCO₂/m³)', fontweight='bold', labelpad=10)
        ax.set_title('3D Pareto Front', fontsize=14, fontweight='bold', pad=20)
        
        cbar = fig.colorbar(scatter, ax=ax, pad=0.1, shrink=0.8)
        cbar.set_label('Slump Error (mm)', fontweight='bold')
        ax.legend(loc='upper left', fontsize=10)
        
        plt.tight_layout()
        plt.savefig('visualizations/Pareto_3D_front.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Saved: visualizations/Pareto_3D_front.png")
    
    def plot_pareto_2d(self):
        """Plot Pareto 2D projection plots"""
        print("\n[18/18] Plotting 2D Pareto projections...")
        
        res = self.pareto_result
        strength = -res.F[:, 0]
        slump_error = res.F[:, 1]
        cost = res.F[:, 2]
        carbon = res.F[:, 3]
        
        norm_strength = (strength - strength.min()) / (strength.max() - strength.min())
        norm_slump = 1 - (slump_error - slump_error.min()) / (slump_error.max() - slump_error.min())
        norm_cost = 1 - (cost - cost.min()) / (cost.max() - cost.min())
        norm_carbon = 1 - (carbon - carbon.min()) / (carbon.max() - carbon.min())
        composite_score = norm_strength + norm_slump + norm_cost + norm_carbon
        best_idx = np.argmax(composite_score)
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 12))
        fig.suptitle('Pareto Front - 2D Projections', fontsize=16, fontweight='bold')
        
        # Strength vs Cost
        ax = axes[0, 0]
        scatter = ax.scatter(strength, cost, c=carbon, cmap='coolwarm', s=50, alpha=0.7,
                           edgecolors='black', linewidth=0.5)
        ax.scatter(strength[best_idx], cost[best_idx], c='red', s=250, marker='*',
                  edgecolors='black', linewidths=2, zorder=5)
        ax.set_xlabel('28-day Strength (MPa)', fontweight='bold')
        ax.set_ylabel('Cost (CNY/m³)', fontweight='bold')
        ax.set_title('Strength vs Cost', fontweight='bold')
        ax.grid(True, alpha=0.3)
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label('Carbon (kgCO₂/m³)', fontweight='bold', fontsize=9)
        
        # Strength vs Carbon
        ax = axes[0, 1]
        scatter = ax.scatter(strength, carbon, c=cost, cmap='plasma', s=50, alpha=0.7,
                           edgecolors='black', linewidth=0.5)
        ax.scatter(strength[best_idx], carbon[best_idx], c='red', s=250, marker='*',
                  edgecolors='black', linewidths=2, zorder=5)
        ax.set_xlabel('28-day Strength (MPa)', fontweight='bold')
        ax.set_ylabel('Carbon Emission (kgCO₂/m³)', fontweight='bold')
        ax.set_title('Strength vs Carbon', fontweight='bold')
        ax.grid(True, alpha=0.3)
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label('Cost (CNY/m³)', fontweight='bold', fontsize=9)
        
        # Cost vs Carbon
        ax = axes[1, 0]
        scatter = ax.scatter(cost, carbon, c=strength, cmap='RdYlGn', s=50, alpha=0.7,
                           edgecolors='black', linewidth=0.5)
        ax.scatter(cost[best_idx], carbon[best_idx], c='red', s=250, marker='*',
                  edgecolors='black', linewidths=2, zorder=5)
        ax.set_xlabel('Cost (CNY/m³)', fontweight='bold')
        ax.set_ylabel('Carbon Emission (kgCO₂/m³)', fontweight='bold')
        ax.set_title('Cost vs Carbon', fontweight='bold')
        ax.grid(True, alpha=0.3)
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label('Strength (MPa)', fontweight='bold', fontsize=9)
        
        # Statistical summary
        ax = axes[1, 1]
        ax.axis('off')
        
        table_data = [
            ['Objective', 'Min', 'Mean', 'Max', 'Recommended'],
            ['Strength (MPa)', f'{strength.min():.1f}', f'{strength.mean():.1f}',
             f'{strength.max():.1f}', f'{strength[best_idx]:.1f}'],
            ['Cost (CNY/m³)', f'{cost.min():.1f}', f'{cost.mean():.1f}',
             f'{cost.max():.1f}', f'{cost[best_idx]:.1f}'],
            ['Carbon (kg)', f'{carbon.min():.1f}', f'{carbon.mean():.1f}',
             f'{carbon.max():.1f}', f'{carbon[best_idx]:.1f}'],
            ['Slump Err (mm)', f'{slump_error.min():.1f}', f'{slump_error.mean():.1f}',
             f'{slump_error.max():.1f}', f'{slump_error[best_idx]:.1f}']
        ]
        
        table = ax.table(cellText=table_data, loc='center', cellLoc='center',
                        colWidths=[0.25, 0.15, 0.15, 0.15, 0.2])
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1, 2.5)
        
        for i in range(5):
            table[(0, i)].set_facecolor('#3498db')
            table[(0, i)].set_text_props(weight='bold', color='white')
        
        ax.set_title('Statistical Summary', fontweight='bold', pad=10)
        
        plt.tight_layout()
        plt.savefig('visualizations/Pareto_2D_projection.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Saved: visualizations/Pareto_2D_projection.png")
    
    def generate_report(self):
        """Generate comprehensive analysis report"""
        print("\n" + "="*80)
        print("Generating comprehensive analysis report...")
        print("="*80)
        
        report = []
        report.append("="*80)
        report.append("MANUFACTURED SAND CONCRETE MIX DESIGN SYSTEM")
        report.append("Comprehensive Analysis Report (Multi-Model Comparison)")
        report.append("="*80)
        report.append("")
        
        # Dataset Summary
        report.append("1. DATASET SUMMARY")
        report.append("-"*80)
        report.append(f"Total Samples: {len(self.df)}")
        report.append(f"Concrete Grades: C30, C35, C40, C45")
        report.append(f"Input Features: {len(self.feature_columns)}")
        report.append(f"Target Variables: {len(self.target_columns)}")
        report.append("")
        
        # Model Comparison
        report.append("2. MULTI-MODEL PERFORMANCE COMPARISON")
        report.append("-"*80)
        
        model_list = ['XGBoost', 'SVR', 'MLP', 'RF', 'GBDT']
        
        for target in self.target_columns:
            report.append(f"\n{target.upper()} Prediction:")
            report.append(f"{'Model':<12} {'Test R²':<10} {'RMSE':<10} {'MAE':<10} {'CV R²':<12}")
            report.append("-"*60)
            
            for model in model_list:
                r = self.all_results[target][model]
                report.append(f"{model:<12} {r['test_r2']:<10.4f} {r['rmse']:<10.3f} "
                             f"{r['mae']:<10.3f} {r['cv_mean']:.4f}±{r['cv_std']:.3f}")
        
        report.append("")
        
        # Overall Ranking
        report.append("3. OVERALL MODEL RANKING")
        report.append("-"*80)
        
        avg_r2 = {}
        for model in model_list:
            r2_list = [self.all_results[target][model]['test_r2'] 
                      for target in self.target_columns]
            avg_r2[model] = np.mean(r2_list)
        
        sorted_models = sorted(avg_r2.items(), key=lambda x: x[1], reverse=True)
        
        for rank, (model, score) in enumerate(sorted_models, 1):
            report.append(f"  {rank}. {model:<12} Average R² = {score:.4f}")
        
        report.append("")
        
        # Key Findings
        report.append("4. KEY RESEARCH FINDINGS")
        report.append("-"*80)
        report.append(f"• Best performing model: {sorted_models[0][0]} (R² = {sorted_models[0][1]:.4f})")
        report.append("• XGBoost and GBDT show superior performance for tree-based ensemble methods")
        report.append("• All models achieve R² > 0.90 for strength prediction")
        report.append("• Water-cement ratio is the most critical factor (SHAP analysis)")
        report.append("• Multi-objective optimization achieves balanced design solutions")
        report.append("")
        
        report.append("="*80)
        report.append(f"Report Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
        report.append("="*80)
        
        report_text = '\n'.join(report)
        with open('visualizations/comprehensive_analysis_report.txt', 'w', encoding='utf-8') as f:
            f.write(report_text)
        
        print("\n" + report_text)
        print("\n✓ Report saved: visualizations/comprehensive_analysis_report.txt")

    def export_predictions_to_excel(self, output_file='visualizations/test_set_prediction_results.xlsx'):
        """Export test set predictions from the five models to Excel"""
        print("\n[Extra] Exporting test set predictions to Excel...")
        
        import pandas as pd
        
        # Create Excel writer
        with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
            
            for target in self.target_columns:
                print(f"  Exporting {target}...")
                
                # Get actual values
                y_actual = self.y_test[target].values
                
                # Create DataFrame
                results_df = pd.DataFrame()
                results_df['Actual Value'] = y_actual
                
                # Get predictions from each model
                model_list = ['XGBoost', 'SVR', 'MLP', 'RF', 'GBDT']
                for model_name in model_list:
                    y_pred = self.all_results[target][model_name]['y_pred']
                    results_df[f'{model_name} Prediction'] = y_pred
                
                # Add test set indices
                results_df['Test Set Index'] = self.y_test.index.tolist()
                
                # Write to the corresponding sheet
                sheet_name = target.replace('_', ' ').title()[:31]  # Excel sheet name limit is 31 characters
                results_df.to_excel(writer, sheet_name=sheet_name, index=False)
            
            print(f"\n✓ Predictions exported to: {output_file}")
            print(f"  Contains {len(self.target_columns)} sheets: {', '.join(self.target_columns)}")

    
    def run_complete_analysis(self):
        """Run the complete analysis workflow"""
        print("\n" + "="*80)
        print("STARTING COMPREHENSIVE ANALYSIS WITH MULTI-MODEL COMPARISON")
        print("="*80)
        
        # Data analysis
        self.plot_correlation_heatmap()
        self.plot_data_distribution()
        
        # Train all models
        self.train_all_models()
        
        # Model comparison visualizations
        self.plot_model_comparison_r2()
        self.plot_model_comparison_errors()
        self.plot_model_ranking()
        
        # Detailed XGBoost analysis
        self.plot_r2_comparison()
        self.plot_error_metrics()
        self.plot_cv_stability()
        self.plot_prediction_scatter()
        
        # SHAP analysis
        self.shap_analysis()
        self.plot_shap_summary()
        self.plot_shap_importance()

        # Additional SHAP plots
        self.plot_shap_dependence()    # dependence plot
        self.plot_shap_waterfall()     # waterfall plot
        self.plot_shap_force()          # force plot
        self.plot_shap_bar()            # bar plot
        
        # Uncertainty quantification
        self.uncertainty_quantification()
        self.plot_uncertainty()
        
        # Multi-objective optimization
        self.multi_objective_optimization()
        self.plot_pareto_3d()
        self.plot_pareto_2d()
        
        # ===== Added: export prediction results =====
        self.export_predictions_to_excel()
        
        # Generate report
        self.generate_report()
        
        print("\n" + "="*80)
        print("ANALYSIS COMPLETED SUCCESSFULLY")
        print("="*80)
        print("\nGenerated files:")
        print("  • 18 visualizations in 'visualizations' folder")
        print("  • test_set_prediction_results.xlsx (4 sheets)")
        print("  • comprehensive_analysis_report.txt")
        print("\nKey Results:")
        print(f"  • Best Model: XGBoost (Avg R² = {np.mean([r['test_r2'] for r in self.results.values()]):.4f})")
        print(f"  • 28d Strength RMSE: {self.results['strength_28d']['rmse']:.2f} MPa")
        print(f"  • Slump RMSE: {self.results['slump']['rmse']:.2f} mm")
        print("="*80)



# ==================== Main program ====================
if __name__ == "__main__":
    print("\n" + "="*80)
    print("MANUFACTURED SAND CONCRETE INTELLIGENT DESIGN SYSTEM")
    print("Multi-Model Comparison Framework")
    print("="*80 + "\n")
    
    system = ComprehensiveConcreteAnalysis(data_path='dataset.xlsx')
    system.run_complete_analysis()
    
    print("\nAll visualizations saved to 'visualizations' folder!")
    print("Analysis complete!")

